In [1]:
import os
import pickle
import datetime
import random

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [2]:
SEED=42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

In [4]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [5]:
X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN Implementation

In [7]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

2026-09-24 13:11:19.215326: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-24 13:11:19.215384: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-24 13:11:19.215404: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-24 13:11:19.215733: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-24 13:11:19.216050: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss="binary_crossentropy",
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ]
)


In [9]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

In [10]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback],
    verbose=1
)


Epoch 1/100


2026-09-24 13:11:20.724927: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-09-24 13:11:20.794602: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


250/250 [==============================] - 7s 22ms/step - loss: 0.4801 - accuracy: 0.7812 - precision: 0.4441 - recall: 0.2926 - auc: 0.7170 - val_loss: 0.4284 - val_accuracy: 0.8065 - val_precision: 0.5595 - val_recall: 0.2310 - val_auc: 0.7676
Epoch 2/100
250/250 [==============================] - 4s 17ms/step - loss: 0.4365 - accuracy: 0.8067 - precision: 0.5654 - recall: 0.2227 - auc: 0.7548 - val_loss: 0.4284 - val_accuracy: 0.8040 - val_precision: 0.5362 - val_recall: 0.2727 - val_auc: 0.7752
Epoch 3/100
250/250 [==============================] - 5s 19ms/step - loss: 0.4382 - accuracy: 0.8087 - precision: 0.5786 - recall: 0.2258 - auc: 0.7518 - val_loss: 0.4269 - val_accuracy: 0.8100 - val_precision: 0.5808 - val_recall: 0.2383 - val_auc: 0.7665
Epoch 4/100
250/250 [==============================] - 4s 17ms/step - loss: 0.4440 - accuracy: 0.8096 - precision: 0.5870 - recall: 0.2215 - auc: 0.7444 - val_loss: 0.4630 - val_accuracy: 0.8005 - val_precision: 0.5299 - val_recall: 0.174

In [11]:
model.save('model.keras')

In [12]:
results = model.evaluate(X_test, y_test, verbose=0)
for metric_name, metric_value in zip(model.metrics_names, results):
    print(f"{metric_name}: {metric_value:.4f}")

loss: 0.4269
accuracy: 0.8100
precision: 0.5808
recall: 0.2383
auc: 0.7665


In [13]:
%load_ext tensorboard

In [15]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6013 (pid 3185), started 0:00:03 ago. (Use '!kill 3185' to kill it.)